In [20]:
import pandas as pd

full_dataset = pd.read_csv("features.csv")

print(full_dataset.head())

train_dataset = full_dataset.sample(frac=0.8, random_state=42)
test_dataset = full_dataset.drop(train_dataset.index)

   packet_size       ttl  protocol  src_port  dst_port  flags  tcp_window  \
0     0.044118  0.196850  0.352941  0.007263  0.799482   0.96    0.001175   
1     0.003443  0.196850  0.352941  0.007263  0.799482   1.00    0.001175   
2     0.001722  0.251969  0.352941  0.799482  0.007263   0.64    0.001236   
3     0.003443  0.251969  0.352941  0.799482  0.007263   0.96    0.001236   
4     0.001722  0.251969  0.352941  0.799482  0.007263   0.68    0.001236   

   payload_size  
0      0.042457  
1      0.001724  
2      0.000000  
3      0.001724  
4      0.000000  


In [21]:
# Write a autoencoder net with binary intermediate layers

import torch
import torch.nn as nn


class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        out = torch.sign(x)
        out[out == 0] = 1
        return out

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        mask = (x.abs() <= 1).float()
        return grad_output * mask


def binarize(x):
    return BinarizeSTE.apply(x)


class BinaryHiddenLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return binarize(self.linear(x))


class BinaryAutoencoder(nn.Module):
    def __init__(self, input_size=8, hidden_size=16, latent_size=4):
        super().__init__()

        self.encoder = nn.Sequential(
            BinaryHiddenLayer(input_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, latent_size),
        )

        self.decoder_hidden = BinaryHiddenLayer(latent_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        latent = self.encoder(x)
        hidden = self.decoder_hidden(latent)
        reconstruction = self.output_layer(hidden)
        return reconstruction, latent


In [ ]:
import numpy as np

x = torch.tensor(train_dataset.to_numpy(dtype=np.float32))

model = BinaryAutoencoder(input_size=x.shape[1], hidden_size=16, latent_size=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 2000
for epoch in range(epochs):
    optimizer.zero_grad()
    reconstruction, latent = model(x)
    loss = torch.nn.functional.mse_loss(reconstruction, x)
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:5d}  loss {loss.item():.6f}")


x_test = torch.tensor(test_dataset.to_numpy(dtype=np.float32))
with torch.no_grad():
    reconstruction, latent = model(x_test)
    print("\nbinary latent codes (first 5 rows):")
    print(latent[:5])
    print("\nreconstruction error (first 5 rows):")
    print((reconstruction - x_test).abs()[:5])


epoch     0  loss 0.482000
epoch   200  loss 0.077409
epoch   400  loss 0.042332
epoch   600  loss 0.071726
epoch   800  loss 0.026768
epoch  1000  loss 0.022163
epoch  1200  loss 0.024433
epoch  1400  loss 0.018742
epoch  1600  loss 0.022279
epoch  1800  loss 0.017928
epoch  1999  loss 0.015767

binary latent codes (first 5 rows):
tensor([[ 1., -1.,  1., -1.],
        [ 1., -1.,  1.,  1.],
        [-1., -1., -1., -1.],
        [ 1.,  1.,  1.,  1.],
        [ 1., -1., -1.,  1.]])

reconstruction error (first 5 rows):
tensor([[0.0720, 0.0423, 0.0502, 0.0964, 0.3507, 0.3104, 0.0132, 0.1149],
        [0.0043, 0.0083, 0.0067, 0.0269, 0.0216, 0.1091, 0.0060, 0.0167],
        [0.0359, 0.0407, 0.0359, 0.0786, 0.0874, 0.0812, 0.0063, 0.0089],
        [0.0202, 0.0925, 0.0051, 0.0765, 0.0729, 0.0873, 0.0202, 0.0291],
        [0.0123, 0.0113, 0.0191, 0.1114, 0.0110, 0.0105, 0.0258, 0.0137]])

binary latent codes (first 5 rows):
tensor([[ 1., -1.,  1.,  1.],
        [ 1.,  1.,  1., -1.],
        [